# Experiment 4: Limited Optimization (RQ4)

Single intervention on 30 high-failure AL families (base + 1 CF each = 60 instances), 3 models (Opus 4.6, Sonnet 4.6, GPT-5-3-Codex):
- **Skill** = AL-structured bug-fix guidance skill (`al-bugfix-structured`), copied to `.github/skills/` (enabled via `skills.enabled` in config.yaml).

Baseline = existing plain CF results. Metrics: correctness gain, layer-specific gain, fragile->stable conversion. The skill result dirs (`*-skill`) are auto-loaded once present; until then only the baseline columns populate.

In [1]:
import csv
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
from utils import load_all_results

BASELINES = {
    "Claude Opus 4.6": "copilot-cf-opus-4-6",
    "Claude Sonnet 4.6": "copilot-cf-sonnet-4-6",
    "GPT-5-3 Codex": "copilot-cf-gpt-5-3-codex",
}

# 30 target families + dominant layer
target = {}
with open("exp4_target_families.csv", encoding="utf-8") as f:
    for r in csv.DictReader(f):
        target[r["family_id"]] = r["dominant_layer"] or "-"

subset = [l.strip() for l in Path("exp4_subset_instances.txt").read_text(encoding="utf-8").splitlines() if l.strip()]
def base_of(iid): return iid.split("__cf-")[0]
layer_of = {iid: target.get(base_of(iid), "-") for iid in subset}

all_results = load_all_results(category="bug-fix")
print(f"{len(subset)} subset instances, {len(target)} families. setups available: {len(all_results)}")

60 subset instances, 30 families. setups available: 7


In [2]:
def resolved_map(setup: str) -> dict[str, bool]:
    """instance_id -> resolved (majority across runs) for the subset, or {} if absent."""
    if setup not in all_results:
        return {}
    df = all_results[setup]
    df = df[df["instance_id"].isin(subset)]
    if df.empty:
        return {}
    return (df.groupby("instance_id")["resolved"].mean() >= 0.5).to_dict()

def skill_setup(base_setup: str) -> str:
    return f"{base_setup}-skill"

print("Skill result dirs present:")
for label, base in BASELINES.items():
    s = skill_setup(base)
    print(f"  {s}: {'YES' if s in all_results else 'not yet'}")

Skill result dirs present:
  copilot-cf-opus-4-6-skill: not yet
  copilot-cf-sonnet-4-6-skill: not yet
  copilot-cf-gpt-5-3-codex-skill: not yet


In [3]:
# Correctness gain: mean resolved on the 60-instance subset, baseline vs skill
rows = []
for label, base in BASELINES.items():
    base_res = resolved_map(base)
    if not base_res:
        continue
    n = len(base_res)
    base_rate = sum(base_res.values()) / n * 100
    row = {"Model": label, "Subset n": n, "Baseline (%)": round(base_rate, 1)}
    sres = resolved_map(skill_setup(base))
    if sres:
        common = [i for i in base_res if i in sres]
        srate = sum(sres[i] for i in common) / len(common) * 100
        row["Skill (%)"] = round(srate, 1)
        row["Gain (pp)"] = round(srate - base_rate, 1)
    else:
        row["Skill (%)"] = None
        row["Gain (pp)"] = None
    rows.append(row)
correctness_gain_df = pd.DataFrame(rows)
correctness_gain_df

,Model,Subset n,Baseline (%),Skill (%),Gain (pp)
0,Claude Opus 4.6,60,31.7,None,None
1,Claude Sonnet 4.6,60,35.0,None,None
2,GPT-5-3 Codex,60,31.7,None,None


In [4]:
# Layer-specific gain: delta resolved grouped by the family's dominant failure layer
layer_rows = []
for label, base in BASELINES.items():
    base_res = resolved_map(base)
    if not base_res:
        continue
    sres = resolved_map(skill_setup(base))
    if not sres:
        continue
    by_layer = {}
    for iid, b in base_res.items():
        if iid not in sres:
            continue
        lay = layer_of.get(iid, "-")
        by_layer.setdefault(lay, []).append(sres[iid] - b)
    for lay, deltas in sorted(by_layer.items()):
        layer_rows.append({
            "Model": label, "Layer": lay,
            "n": len(deltas), "Gain (pp)": round(sum(deltas) / len(deltas) * 100, 1),
        })
layer_gain_df = pd.DataFrame(layer_rows)
layer_gain_df if not layer_gain_df.empty else "No skill results yet."

'No skill results yet.'

In [5]:
# Fragile/unsolved -> stable conversion, per family (base + its CFs in the subset)
from bcbench.analysis.aggregator import build_families
from bcbench.analysis.family import FamilyType
from bcbench.results.base import ExecutionBasedEvaluationResult

def families_on_subset(setup: str):
    if setup not in all_results:
        return {}
    df = all_results[setup]
    df = df[df["instance_id"].isin(subset)]
    res = [ExecutionBasedEvaluationResult(instance_id=r["instance_id"], project="", model=setup,
            agent_name="x", category="cf", resolved=bool(r["resolved"]), build=bool(r.get("build", False)),
            output=r.get("output", "")) for _, r in df.iterrows()]
    return {f.family_id: f.family_type for f in build_families(res)}

conv_rows = []
for label, base in BASELINES.items():
    base_fam = families_on_subset(base)
    if not base_fam:
        continue
    sfam = families_on_subset(skill_setup(base))
    if not sfam:
        continue
    converted = sum(1 for fid, t in base_fam.items()
                    if t != FamilyType.STABLE_CORRECT
                    and sfam.get(fid) == FamilyType.STABLE_CORRECT)
    eligible = sum(1 for t in base_fam.values() if t != FamilyType.STABLE_CORRECT)
    conv_rows.append({"Model": label,
                      "Non-stable families": eligible, "-> stable": converted,
                      "Conversion (%)": round(converted / eligible * 100, 1) if eligible else None})
conversion_df = pd.DataFrame(conv_rows)
conversion_df if not conversion_df.empty else "No skill results yet."

'No skill results yet.'